## Experiment Generator

In [ ]:
import os
import gc
import warnings
import torch
import pandas as pd
import numpy as np
from models.tempo import MultiModalDeepSurv
from models.trainer import Trainer
from captum.attr import IntegratedGradients
warnings.filterwarnings('ignore')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
slice_num = 5
batch_size = 32
lr = 0.0005
epochs = 50
ablation = "t2"
pre_train_resnet_path = "../TEMPO/dataset/MRI_pretrained.pth"
tab_df_path = "../TEMPO/dataset/tabular_input.csv"
checkpoint_path = f"output/checkpoint_{ablation}.pth"
dataset_type = "testset"  # [trainset, testset]
target_folder = f"Experiments/{ablation}/"
if os.path.exists(target_folder):
    None
else:
    os.mkdir(target_folder)

In [ ]:
tempo = Trainer(slice_num, batch_size, lr, epochs, ablation, pre_train_resnet_path, tab_df_path)
tab_df, trainset, train_set, trainloader, testset, test_set, testloader = tempo.data_load()
model = MultiModalDeepSurv(
    feature_dim = 16,
            fusion_dim = train_set.get_params()["tabular_shape"],
            middle_dim = 8,
            ablation = ablation, # only for pret
).to(device)
checkpoint = torch.load(checkpoint_path,weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
if dataset_type == "testset":
    data_standard = testloader
else:
    data_standard = trainloader

In [ ]:
def weighted_mean_func(group):
    # 以 y_duration 作为权重
    weights = group['y_duration']
    total_weight = weights.sum()

    # 计算加权平均的逻辑
    if total_weight == 0:
        risk_weighted = group['risk_score_pred'].mean()
        prog_weighted = group['prog_pred'].mean()
    else:
        risk_weighted = (group['risk_score_pred'] * weights).sum() / total_weight
        prog_weighted = (group['prog_pred'] * weights).sum() / total_weight

    return pd.Series({
        'y_duration': group['y_duration'].max(),
        'y_event': group['y_event'].max(),
        'y_prog': group['y_prog'].max(),
        'risk_score_pred': risk_weighted,
        'prog_pred': prog_weighted
    })

In [ ]:
results_list = []

model.eval()
with torch.no_grad():
    for batch in data_standard:
        x_input = batch["x_input"].to(device)
        y_duration = batch["y_duration"].to(device)
        y_event = batch["y_event"].to(device)
        y_prog = batch["y_prog_risk"].to(device)
        axi_img = batch["axi_img"].to(device)
        cor_img = batch["cor_img"].to(device)
        sag_img = batch["sag_img"].to(device)
        mic_img = batch["mic_img"].to(device)
        patient_id = batch["patient_id"]

        risk_score, prog_pred = model(axi_img, cor_img, sag_img, mic_img, x_input)
        batch_df = pd.DataFrame({
            'patient_id': patient_id.cpu().numpy().flatten() if torch.is_tensor(patient_id) else patient_id,
            'y_duration': y_duration.cpu().numpy().flatten(),
            'y_event': y_event.cpu().numpy().flatten(),
            'risk_score_pred': risk_score.cpu().numpy().flatten(),
            'prog_pred': prog_pred.cpu().numpy().flatten(),
            'y_prog': y_prog.cpu().numpy().flatten(),

        })

        results_list.append(batch_df)

data = pd.concat(results_list, ignore_index=True)
data_agg = data.groupby('patient_id').apply(weighted_mean_func).reset_index()
data.to_csv(f"{target_folder}{dataset_type}_indiv_output.csv",index=False)
data_agg.to_csv(f"{target_folder}{dataset_type}_output.csv",index=False)

### Contribution Analysis

In [ ]:
class IGWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, axi, cor, sag, mic, x_tab):
        risk_score,_ = self.model(axi, cor, sag, mic, x_tab)
        return risk_score

idx = 0
input_data = (
    batch["axi_img"][idx:idx+1].to(device).requires_grad_(),
    batch["cor_img"][idx:idx+1].to(device).requires_grad_(),
    batch["sag_img"][idx:idx+1].to(device).requires_grad_(),
    batch["mic_img"][idx:idx+1].to(device).requires_grad_(),
    batch["x_input"][idx:idx+1].to(device).requires_grad_()
)


wrapped_model = IGWrapper(model).eval()
ig = IntegratedGradients(wrapped_model)
torch.cuda.empty_cache()
gc.collect()
attributions = ig.attribute(inputs=input_data, n_steps=5)
axi_attr = attributions[0].detach().cpu().numpy()
tab_attr = attributions[4].detach().cpu().numpy()

In [ ]:
cate_cols = test_set.get_params()["categorical_features"]
cont_cols = test_set.get_params()["continuous_features"]
cate_cols = test_set.get_params()["cate_scaler"].get_feature_names_out(cate_cols).tolist()
features = cont_cols + cate_cols
raw_attr = tab_attr.flatten()

In [ ]:
df_plot = pd.DataFrame({
    'feature': features,
    'attribution': raw_attr,
    'abs_importance': np.abs(raw_attr)
})

df_plot = df_plot[df_plot['attribution'] != 0].copy()
df_plot.to_csv(f"{target_folder}{dataset_type}_contributor.csv",index=False)